# Oracle evaluation
Loads the best checkpoint, runs inference on val and test sets, and compares to SAIR baselines.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
os.environ.setdefault('PYTORCH_MPS_HIGH_WATERMARK_RATIO', '0.0')

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.stats import spearmanr, pearsonr
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

from oracle import IC50Oracle, SAIRDataset
from scripts.train import collate_fn

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

In [ ]:
CKPT_PATH   = Path('../checkpoints/best.pt')
PARQUET     = Path('../data/sair.parquet')
ESM_CACHE   = Path('../data/esm_embeddings/')
ANNOTATIONS = Path('../data/long_protein_annotations.csv')
SPLITS_DIR  = Path('../data/splits/')
BATCH_SIZE  = 512

device = (
    torch.device('cuda') if torch.cuda.is_available()
    else torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cpu')
)
print(f'Device: {device}')

## 1. Load model from checkpoint

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model = IC50Oracle.from_config(ckpt['config']).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

print(f"Checkpoint: epoch {ckpt['epoch']}  val_spearman={ckpt['val_spearman']:.4f}  val_mae={ckpt['val_mae']:.4f}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Build val and test datasets

In [ ]:
def load_ids(split):
    return [(SPLITS_DIR / f'{split}.txt').read_text().splitlines()][0]

dataset_kwargs = dict(
    parquet_path=PARQUET,
    esm_cache_dir=ESM_CACHE,
    annotations_csv=ANNOTATIONS,
    filter_all_passed=True,
    assay_filter='biochem',
    deduplicate=True,
)

val_ds  = SAIRDataset(entry_ids=load_ids('val'),  **dataset_kwargs)
test_ds = SAIRDataset(entry_ids=load_ids('test'), **dataset_kwargs)
print(f'Val:  {len(val_ds):,} samples  |  Test: {len(test_ds):,} samples')

## 3. Run inference

In [ ]:
@torch.no_grad()
def run_inference(dataset, batch_size=BATCH_SIZE):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=0, collate_fn=collate_fn)
    preds, targets, proteins = [], [], []
    for protein_embs, graphs, pic50s, protein_ids in tqdm(loader):
        protein_embs = protein_embs.to(device)
        graphs = graphs.to(device)
        p = model(protein_embs, graphs).float().cpu()
        preds.append(p)
        targets.append(pic50s)
        proteins.append(protein_ids)
        if device.type == 'mps':
            torch.mps.empty_cache()

    # Build a results dataframe aligned with dataset.df index
    df = dataset.df[['entry_id', 'protein', 'SMILES', 'pIC50']].copy().reset_index(drop=True)
    df['pred_pIC50'] = torch.cat(preds).numpy()
    return df

val_df  = run_inference(val_ds)
test_df = run_inference(test_ds)
print('Done.')

In [ ]:
# Merge in baseline scores and protein family from parquet
parquet_cols = ['entry_id', 'family', 'vina_score', 'vinardo_score',
                'aevplig_score', 'onionnet_score', 'vina_score_min']
meta = pd.read_parquet(PARQUET, columns=parquet_cols)

val_df  = val_df.merge(meta,  on='entry_id', how='left')
test_df = test_df.merge(meta, on='entry_id', how='left')
val_df.head(3)

## 4. Overall metrics

In [ ]:
def metrics(df, pred_col, target_col='pIC50', label=''):
    mask = df[pred_col].notna() & df[target_col].notna()
    p = df.loc[mask, pred_col].values
    t = df.loc[mask, target_col].values
    return {
        'model': label,
        'n': mask.sum(),
        'spearman': spearmanr(p, t).statistic,
        'pearson':  pearsonr(p, t)[0],
        'mae':      float(np.abs(p - t).mean()),
        'rmse':     float(np.sqrt(((p - t)**2).mean())),
    }

# Docking scores are negative (lower = better) — negate for fair Spearman comparison
for df in [val_df, test_df]:
    for col in ['vina_score', 'vinardo_score', 'vina_score_min']:
        df[f'{col}_neg'] = -df[col]

rows = []
for split_label, df in [('val', val_df), ('test', test_df)]:
    rows.append({**metrics(df, 'pred_pIC50',         label='Oracle'),        'split': split_label})
    rows.append({**metrics(df, 'aevplig_score',      label='AEVPLig'),       'split': split_label})
    rows.append({**metrics(df, 'onionnet_score',     label='OnionNet'),      'split': split_label})
    rows.append({**metrics(df, 'vina_score_neg',     label='Vina (neg)'),    'split': split_label})
    rows.append({**metrics(df, 'vinardo_score_neg',  label='Vinardo (neg)'), 'split': split_label})

metrics_df = pd.DataFrame(rows)
metrics_df = metrics_df[['split', 'model', 'n', 'spearman', 'pearson', 'mae', 'rmse']]
print(metrics_df.to_string(index=False, float_format='{:.4f}'.format))

## 5. Predicted vs actual scatter (coloured by protein family)

In [ ]:
FAMILY_COLORS = {
    'kinase':           '#e74c3c',
    'enzyme':           '#3498db',
    'GPCR':             '#2ecc71',
    'nuclear receptor': '#f39c12',
    'phosphatase':      '#9b59b6',
    'other':            '#95a5a6',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (split_label, df) in zip(axes, [('Val', val_df), ('Test', test_df)]):
    sample = df.sample(min(5000, len(df)), random_state=42)
    for family, grp in sample.groupby('family'):
        ax.scatter(grp['pIC50'], grp['pred_pIC50'],
                   c=FAMILY_COLORS.get(family, '#95a5a6'),
                   alpha=0.35, s=8, label=family, rasterized=True)
    lo, hi = df['pIC50'].min() - 0.3, df['pIC50'].max() + 0.3
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1, alpha=0.6)
    sp = spearmanr(df['pIC50'], df['pred_pIC50']).statistic
    ax.set_title(f'{split_label}  (Spearman={sp:.3f})')
    ax.set_xlabel('True pIC50')
    ax.set_ylabel('Predicted pIC50')
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)

handles = [plt.Line2D([0],[0], marker='o', color='w',
           markerfacecolor=c, markersize=7, label=f)
           for f, c in FAMILY_COLORS.items()]
axes[1].legend(handles=handles, title='Family', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../checkpoints/scatter_val_test.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Per-protein Spearman distribution

In [ ]:
def per_protein_spearman(df, pred_col='pred_pIC50', target_col='pIC50', min_samples=5):
    rows = []
    for protein, grp in df.groupby('protein'):
        if len(grp) < min_samples:
            continue
        sp = spearmanr(grp[target_col], grp[pred_col]).statistic
        rows.append({'protein': protein, 'spearman': sp,
                     'n': len(grp), 'family': grp['family'].iloc[0]})
    return pd.DataFrame(rows).sort_values('spearman', ascending=False)

val_pp  = per_protein_spearman(val_df)
test_pp = per_protein_spearman(test_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (label, pp) in zip(axes, [('Val', val_pp), ('Test', test_pp)]):
    ax.hist(pp['spearman'], bins=30, edgecolor='white', color='#3498db', alpha=0.85)
    ax.axvline(pp['spearman'].median(), color='k', linestyle='--', linewidth=1.2,
               label=f'median={pp["spearman"].median():.3f}')
    ax.set_title(f'{label} — per-protein Spearman ({len(pp):,} proteins ≥5 samples)')
    ax.set_xlabel('Spearman ρ')
    ax.set_ylabel('Count')
    ax.legend()
plt.tight_layout()
plt.savefig('../checkpoints/per_protein_spearman.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Val  — mean: {val_pp["spearman"].mean():.4f}  '
      f'median: {val_pp["spearman"].median():.4f}  '
      f'<0: {(val_pp["spearman"]<0).sum()} proteins')
print(f'Test — mean: {test_pp["spearman"].mean():.4f}  '
      f'median: {test_pp["spearman"].median():.4f}  '
      f'<0: {(test_pp["spearman"]<0).sum()} proteins')

## 7. Best and worst proteins (test set)

In [ ]:
top10    = test_pp.head(10)[['protein', 'family', 'n', 'spearman']]
bottom10 = test_pp.tail(10)[['protein', 'family', 'n', 'spearman']]

print('── Top 10 proteins (test) ──')
print(top10.to_string(index=False))
print()
print('── Bottom 10 proteins (test) ──')
print(bottom10.to_string(index=False))

In [ ]:
# Scatter for the 2 best and 2 worst proteins
proteins_to_plot = test_pp.head(2)['protein'].tolist() + test_pp.tail(2)['protein'].tolist()
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, prot in zip(axes, proteins_to_plot):
    grp = test_df[test_df['protein'] == prot]
    sp  = spearmanr(grp['pIC50'], grp['pred_pIC50']).statistic
    ax.scatter(grp['pIC50'], grp['pred_pIC50'], s=18, alpha=0.6)
    lo = min(grp['pIC50'].min(), grp['pred_pIC50'].min()) - 0.3
    hi = max(grp['pIC50'].max(), grp['pred_pIC50'].max()) + 0.3
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1)
    ax.set_title(f'{prot}\nρ={sp:.3f}  n={len(grp)}')
    ax.set_xlabel('True pIC50')
    ax.set_ylabel('Predicted pIC50')

plt.tight_layout()
plt.savefig('../checkpoints/best_worst_proteins.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Baseline comparison — oracle vs docking scores

In [ ]:
test_metrics = metrics_df[metrics_df['split'] == 'test'].drop(columns='split')
test_metrics = test_metrics.sort_values('spearman', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#e74c3c' if m == 'Oracle' else '#95a5a6' for m in test_metrics['model']]
bars = ax.barh(test_metrics['model'], test_metrics['spearman'], color=colors, edgecolor='white')
ax.bar_label(bars, fmt='{:.3f}', padding=3)
ax.set_xlabel('Spearman ρ  (vs true pIC50, test set)')
ax.set_title('Oracle vs SAIR baselines')
ax.set_xlim(0, max(test_metrics['spearman'].max() + 0.1, 0.5))
plt.tight_layout()
plt.savefig('../checkpoints/baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(test_metrics[['model','spearman','pearson','mae','rmse']].to_string(index=False, float_format='{:.4f}'.format))

## 9. Calibration — pIC50 distribution on held-out SMILES
Sanity check: oracle predictions on SMILES never seen in training should produce a distribution centred on the drug-like range (7–9).

In [ ]:
from oracle.featurise import smiles_to_graph, fallback_graph
from torch_geometric.data import Batch

# Load SMILES not in training set
train_ids  = set((SPLITS_DIR / 'train.txt').read_text().splitlines())
val_ids    = set((SPLITS_DIR / 'val.txt').read_text().splitlines())
test_ids   = set((SPLITS_DIR / 'test.txt').read_text().splitlines())
seen_ids   = train_ids | val_ids | test_ids

holdout = pd.read_parquet(PARQUET, columns=['entry_id', 'protein', 'SMILES', 'pIC50', 'all_passed', 'assay'])
holdout = holdout[
    (holdout['all_passed'] == True) &
    (holdout['assay'] == 'biochem') &
    (~holdout['entry_id'].astype(str).isin(seen_ids))
].dropna(subset=['SMILES', 'pIC50'])

# Only proteins for which we have embeddings
embedded = {p.stem for p in ESM_CACHE.glob('*.pt')}
holdout  = holdout[holdout['protein'].isin(embedded)].sample(
    min(2000, len(holdout)), random_state=42
).reset_index(drop=True)
print(f'Held-out sample: {len(holdout):,} rows')

In [ ]:
@torch.no_grad()
def predict_batch(proteins, smiles_list, batch_size=256):
    all_preds = []
    for start in range(0, len(proteins), batch_size):
        prots = proteins[start:start + batch_size]
        smls  = smiles_list[start:start + batch_size]
        embs  = torch.stack([torch.load(ESM_CACHE / f'{p}.pt', weights_only=True) for p in prots]).to(device)
        graphs = Batch.from_data_list(
            [smiles_to_graph(s) or fallback_graph() for s in smls]
        ).to(device)
        preds = model(embs, graphs).float().cpu()
        all_preds.append(preds)
        if device.type == 'mps':
            torch.mps.empty_cache()
    return torch.cat(all_preds).numpy()

holdout['pred_pIC50'] = predict_batch(
    holdout['protein'].tolist(), holdout['SMILES'].tolist()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(holdout['pIC50'],      bins=40, alpha=0.7, label='True pIC50',      color='#3498db', edgecolor='white')
axes[0].hist(holdout['pred_pIC50'], bins=40, alpha=0.7, label='Predicted pIC50', color='#e74c3c', edgecolor='white')
axes[0].set_title('pIC50 distribution — held-out SMILES')
axes[0].set_xlabel('pIC50')
axes[0].set_ylabel('Count')
axes[0].legend()

sp = spearmanr(holdout['pIC50'], holdout['pred_pIC50']).statistic
axes[1].scatter(holdout['pIC50'], holdout['pred_pIC50'], s=10, alpha=0.4, rasterized=True)
lo = holdout['pIC50'].min() - 0.3
hi = holdout['pIC50'].max() + 0.3
axes[1].plot([lo, hi], [lo, hi], 'k--', linewidth=1)
axes[1].set_title(f'Held-out scatter  (Spearman={sp:.3f})')
axes[1].set_xlabel('True pIC50')
axes[1].set_ylabel('Predicted pIC50')

plt.tight_layout()
plt.savefig('../checkpoints/calibration_holdout.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Held-out Spearman: {sp:.4f}')
print(f'Pred range: [{holdout["pred_pIC50"].min():.2f}, {holdout["pred_pIC50"].max():.2f}]')
print(f'True range: [{holdout["pIC50"].min():.2f}, {holdout["pIC50"].max():.2f}]')